 Number of pages 

In [54]:
from pathlib import Path
import os
from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / '.env')

pdf_path = os.getenv('PDF_PATH')
if not pdf_path:
    raise ValueError('PDF_PATH must be set in the .env file')
pdf_path = Path(pdf_path)
if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

# import fitz  # PyMuPDF

# doc = fitz.open(pdf_path)

# print('PDF path:', pdf_path)
# print('Total Pages:', len(doc))
# print('-' * 40)

# # Loop through each page
# for page_num in range(len(doc)):
#     page = doc[page_num]
#     text = page.get_text('text')  # extract readable text

#     print(f"\n Page {page_num + 1}")
#     #     if text.strip():
#         print(text[:1000])  # show first 1000 chars (avoid huge output)
#     else:
#         print(' No text found on this page (maybe image-based PDF)')

import re
import fitz

doc = fitz.open(pdf_path)

for pdf_page_num in range(len(doc)):
    page = doc[pdf_page_num]
    text = page.get_text("text")

    # Look for standalone page numbers
    lines = text.splitlines()

    book_page = None

    for line in reversed(lines):  # page numbers are often near bottom
        line = line.strip()

        if re.fullmatch(r"\d+", line):
            book_page = int(line)
            break

    print(
        f"PDF Page {pdf_page_num + 1} -> "
        f"Book Page {book_page if book_page else 'Not Found'}"
    )


PDF Page 1 -> Book Page Not Found
PDF Page 2 -> Book Page Not Found
PDF Page 3 -> Book Page Not Found
PDF Page 4 -> Book Page Not Found
PDF Page 5 -> Book Page Not Found
PDF Page 6 -> Book Page 2012041233
PDF Page 7 -> Book Page Not Found
PDF Page 8 -> Book Page Not Found
PDF Page 9 -> Book Page Not Found
PDF Page 10 -> Book Page Not Found
PDF Page 11 -> Book Page Not Found
PDF Page 12 -> Book Page Not Found
PDF Page 13 -> Book Page Not Found
PDF Page 14 -> Book Page Not Found
PDF Page 15 -> Book Page Not Found
PDF Page 16 -> Book Page Not Found
PDF Page 17 -> Book Page 2
PDF Page 18 -> Book Page 4
PDF Page 19 -> Book Page 6
PDF Page 20 -> Book Page 8
PDF Page 21 -> Book Page 1
PDF Page 22 -> Book Page 3
PDF Page 23 -> Book Page 5
PDF Page 24 -> Book Page 7
PDF Page 25 -> Book Page 9
PDF Page 26 -> Book Page Not Found
PDF Page 27 -> Book Page Not Found
PDF Page 28 -> Book Page Not Found
PDF Page 29 -> Book Page Not Found
PDF Page 30 -> Book Page Not Found
PDF Page 31 -> Book Page 1
PDF

Extract Text Page By Page

In [45]:
pages_text = []

for page_num in range(len(doc)):
    
    page = doc.load_page(page_num)
    
    text = page.get_text()

    pages_text.append({
        "page": page_num + 1,
        "text": text
    })

print("Pages Extracted:", len(pages_text))

Pages Extracted: 1170


Inspect Pages

In [46]:
for i, page in enumerate(pages_text, start=1):

    print("_" * 50)
    print(f"PAGE: {i}")
    print("_" * 50)

    text = page.get("text", "").strip()

    if text:
        print(text[:2000])
    else:
        print("No text found on this page")

__________________________________________________
PAGE: 1
__________________________________________________
No text found on this page
__________________________________________________
PAGE: 2
__________________________________________________
No text found on this page
__________________________________________________
PAGE: 3
__________________________________________________
Seventh Edition
Introduction
1  Thorax 
2  Abdomen
3  Pelvis and Perineum
4  Back
5  Lower Limb
6  Upper Limb
7  Head
8  Neck
9  Cranial Nerves
__________________________________________________
PAGE: 4
__________________________________________________
No text found on this page
__________________________________________________
PAGE: 5
__________________________________________________
Keith L. Moore, M.Sc., Ph.D., D.Sc. (Hon), 
F.I.A.C., F.R.S.M., F.A.A.A.
Professor Emeritus in Division of Anatomy, Department of Surgery
Former Chair of Anatomy and Associate
Dean for Basic Medical Sciences
Faculty of Medici

Detect Chapters

In [64]:
import re

chapter_pattern = r'Chapter\s+\d+.*'

chapters = []

for page in pages_text:

    matches = re.findall(
        chapter_pattern,
        page["text"],
        re.IGNORECASE
    )

    for m in matches:

        chapters.append({
            "chapter": m.strip(),
            "page": page["page"]
        })

chapters




[{'chapter': 'CHAPTER 1', 'page': 27},
 {'chapter': 'CHAPTER 2', 'page': 27},
 {'chapter': 'CHAPTER 3', 'page': 28},
 {'chapter': 'CHAPTER 4', 'page': 28},
 {'chapter': 'CHAPTER 6', 'page': 29},
 {'chapter': 'CHAPTER 5', 'page': 29},
 {'chapter': 'CHAPTER 8', 'page': 30},
 {'chapter': 'CHAPTER 9', 'page': 30},
 {'chapter': 'CHAPTER 7', 'page': 30},
 {'chapter': 'Chapter 9 also presents systemic anatomy in reviewing the',
  'page': 34},
 {'chapter': 'Chapter 2), and any surgeon operating with-', 'page': 42},
 {'chapter': 'Chapter 1).', 'page': 44},
 {'chapter': 'Chapter 7).', 'page': 53},
 {'chapter': 'CHAPTER\n1', 'page': 101},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 102},
 {'chapter': 'Chapter 6) that overlie the thoracic cage and form the bed of',
  'page': 102},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 103},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 104},
 {'chapter': 'Chapter 4). The', 'page': 104},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 105},
 {'chapter': 'Chapte

Calculate Chapter End Pages

In [48]:
for i in range(len(chapters)):

    start_page = chapters[i]["page"]

    if i < len(chapters)-1:
        end_page = chapters[i+1]["page"] - 1
    else:
        end_page = len(doc)

    chapters[i]["end_page"] = end_page

chapters

[{'chapter': 'CHAPTER 1', 'page': 27, 'end_page': 26},
 {'chapter': 'CHAPTER 2', 'page': 27, 'end_page': 27},
 {'chapter': 'CHAPTER 3', 'page': 28, 'end_page': 27},
 {'chapter': 'CHAPTER 4', 'page': 28, 'end_page': 28},
 {'chapter': 'CHAPTER 6', 'page': 29, 'end_page': 28},
 {'chapter': 'CHAPTER 5', 'page': 29, 'end_page': 29},
 {'chapter': 'CHAPTER 8', 'page': 30, 'end_page': 29},
 {'chapter': 'CHAPTER 9', 'page': 30, 'end_page': 29},
 {'chapter': 'CHAPTER 7', 'page': 30, 'end_page': 33},
 {'chapter': 'Chapter 9 also presents systemic anatomy in reviewing the',
  'page': 34,
  'end_page': 100},
 {'chapter': 'CHAPTER\n1', 'page': 101, 'end_page': 101},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 102, 'end_page': 101},
 {'chapter': 'Chapter 6) that overlie the thoracic cage and form the bed of',
  'page': 102,
  'end_page': 102},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 103, 'end_page': 103},
 {'chapter': 'Chapter 1  •  Thorax', 'page': 104, 'end_page': 104},
 {'chapter': 'Chapter 

Detect Topics

In [49]:
topic_pattern = r'\d+\.\d+\s+[A-Za-z].*'
topics = []
for page in pages_text:

    matches = re.findall(
        topic_pattern,
        page["text"]
    )

    for m in matches:

        topics.append({
            "topic": m.strip(),
            "page": page["page"]
        })

topics[:20]

[{'topic': '1.48 Adapted with permission from Moore KL, Persaud TVN.',
  'page': 27},
 {'topic': '1.50 Adapted with permission from Torrent-Guasp F, Buckberg',
  'page': 27},
 {'topic': '1.7 Based on Hall-Craggs ECB: Anatomy as the Basis of Clini-',
  'page': 27},
 {'topic': '1.9 Based on Stedman’s Medical Dictionary. 27th ed. (artist:',
  'page': 27},
 {'topic': '1.13 Stedman’s Medical Dictionary. 27th ed. (artist: Neil O.',
  'page': 27},
 {'topic': '1.14 Clinical Radiology: The Essentials. 2nd ed.', 'page': 27},
 {'topic': '1.18 Based on Stedman’s Medical Dictionary. 27th ed. (artist:',
  'page': 27},
 {'topic': '1.19 With permission from The Developing Human: Clinically',
  'page': 27},
 {'topic': '14.15 and 14.14, p. 345–346.', 'page': 27},
 {'topic': '1.23 Siemens Medical Solutions USA, Inc.', 'page': 27},
 {'topic': '1.26 Based on ﬁ', 'page': 27},
 {'topic': '1.28 Based on Stedman’s Medical Dictionary. 27th ed. (artist:',
  'page': 27},
 {'topic': '1.29 Based on Stedman’s Medica

Assign Topics To Chapters

In [50]:
for chapter in chapters:

    chapter["topics"] = []

    start = chapter["page"]
    end = chapter["end_page"]

    for topic in topics:

        if start <= topic["page"] <= end:

            chapter["topics"].append(topic)

chapters

[{'chapter': 'CHAPTER 1', 'page': 27, 'end_page': 26, 'topics': []},
 {'chapter': 'CHAPTER 2',
  'page': 27,
  'end_page': 27,
  'topics': [{'topic': '1.48 Adapted with permission from Moore KL, Persaud TVN.',
    'page': 27},
   {'topic': '1.50 Adapted with permission from Torrent-Guasp F, Buckberg',
    'page': 27},
   {'topic': '1.7 Based on Hall-Craggs ECB: Anatomy as the Basis of Clini-',
    'page': 27},
   {'topic': '1.9 Based on Stedman’s Medical Dictionary. 27th ed. (artist:',
    'page': 27},
   {'topic': '1.13 Stedman’s Medical Dictionary. 27th ed. (artist: Neil O.',
    'page': 27},
   {'topic': '1.14 Clinical Radiology: The Essentials. 2nd ed.', 'page': 27},
   {'topic': '1.18 Based on Stedman’s Medical Dictionary. 27th ed. (artist:',
    'page': 27},
   {'topic': '1.19 With permission from The Developing Human: Clinically',
    'page': 27},
   {'topic': '14.15 and 14.14, p. 345–346.', 'page': 27},
   {'topic': '1.23 Siemens Medical Solutions USA, Inc.', 'page': 27},
   {'